# Lab 3: Deploying MCP Servers to Azure

In this lab, you'll deploy an MCP server to the cloud and connect it to Azure AI Foundry.

## Why Deploy Remotely?

| Local (STDIO) | Remote (HTTP/HTTPS) |
|---------------|--------------------|
| Single machine | Accessible from anywhere |
| One client at a time | Multiple concurrent clients |
| No auth needed | Auth + TLS required |
| Great for personal tools | Great for team/shared tools |

## What We'll Build

```
┌─────────────┐     Docker      ┌───────────────┐     HTTPS      ┌──────────────┐
│ MCP Server  │────────────────►│ Azure         │◄──────────────│ AI Foundry   │
│ (Python)    │     Build       │ Container App │               │ Agent        │
└─────────────┘                 └───────────────┘               └──────────────┘
```

---
## Part 1: The Server

Let's examine the MCP server we'll deploy. It has 4 tools designed for remote use.

In [ ]:
# Read and display the server code
from pathlib import Path

server_code = Path("server/azure_mcp_server.py").read_text()
print(server_code)

### Key differences from Lab 1:

1. **HTTP transport** — `mcp.run(transport="streamable-http", ...)` instead of `stdio`
2. **Async tools** — Uses `async def` for network operations (non-blocking)
3. **External API calls** — Tools call GitHub API, external URLs
4. **Designed for remote** — Tools that benefit from server-side execution

### Let's test the tools locally first

In [ ]:
import httpx
import time

# Test check_service_health logic
async def check_service_health(urls: str) -> str:
    results = []
    async with httpx.AsyncClient(follow_redirects=True, timeout=10.0) as client:
        for url in urls.split(","):
            url = url.strip()
            try:
                start = time.monotonic()
                resp = await client.get(url)
                elapsed = (time.monotonic() - start) * 1000
                results.append(f"  {resp.status_code}  {elapsed:6.0f}ms  {url}")
            except Exception as e:
                results.append(f"  ERROR  {url}: {e}")
    return "\n".join(results)

result = await check_service_health("https://httpbin.org/get,https://httpbin.org/status/404")
print(result)

In [ ]:
# Test run_query logic
sample_data = [
    {"id": 1, "type": "service", "name": "api-gateway", "status": "healthy", "region": "eastus"},
    {"id": 3, "type": "service", "name": "user-service", "status": "degraded", "region": "westus"},
    {"id": 8, "type": "service", "name": "notification-svc", "status": "stopped", "region": "westus"},
]

query = "degraded"
matches = [item for item in sample_data if any(query.lower() in str(v).lower() for v in item.values())]
for m in matches:
    print(f"  [{m['type']}] {m['name']} — {m['status']} ({m['region']})")

---
## Part 2: Containerize with Docker

### The Dockerfile

```dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY azure_mcp_server.py .
EXPOSE 9000
CMD ["python", "azure_mcp_server.py"]
```

### Build and test

Run these in your terminal (not in the notebook):

```bash
cd labs/lab3-azure-deployment/server

# Build
docker build -t mcp-server:latest .

# Run
docker run -p 9000:9000 mcp-server:latest

# Test (in another terminal)
curl http://localhost:9000/docs
```

---
## Part 3: Deploy to Azure

Follow the detailed steps in **[deploy/deploy_steps.md](deploy/deploy_steps.md)**.

### Quick version (automated)

```bash
./labs/lab3-azure-deployment/deploy/deploy.sh
```

### What it creates

```
Azure Resources Created:
├── Resource Group (rg-mcp-hol)
│   ├── Container Registry (mcpholregistry)
│   ├── Container Apps Environment (mcp-env)
│   ├── Container App (mcp-server-app)  ← Your MCP server
│   │   └── https://<fqdn>:443 → :9000
│   ├── AI Foundry Hub (mcp-hol-hub)
│   └── AI Foundry Project (mcp-hol-project)
│       └── Agent with MCP tools
```

### For observers

If you don't have an Azure subscription, follow along with the screenshots and expected outputs in `deploy_steps.md`. Each step shows what you would see.

---
## Part 4: Connect to AI Foundry

Azure AI Foundry agents can use MCP servers as tools. The agent:

1. Discovers available tools from the MCP server
2. Decides which tools to call based on the user's request
3. Calls the tools via HTTPS
4. Incorporates the results into its response

### Setup

Follow **[foundry/foundry_setup.md](foundry/foundry_setup.md)** for detailed instructions.

### Quick version

```bash
# Set environment variables
export AZURE_AI_PROJECT_ENDPOINT="https://<hub>.services.ai.azure.com/api/projects/<id>"
export MCP_SERVER_URL="https://<fqdn>/mcp"

# Run the agent
python labs/lab3-azure-deployment/foundry/foundry_agent.py
```

---
## Part 5: Cleanup

**Important**: Delete your Azure resources after the lab to avoid ongoing charges.

```bash
# Delete everything at once
az group delete --name rg-mcp-hol --yes --no-wait

# Or use the cleanup script
./labs/lab3-azure-deployment/deploy/cleanup.sh
```

---
## Summary

| Concept | What You Learned |
|---------|------------------|
| HTTP Transport | MCP servers accessible over the network |
| Docker | Containerizing Python MCP servers |
| Azure Container Apps | Hosting containers with HTTPS endpoints |
| Azure Container Registry | Storing Docker images in Azure |
| Azure AI Foundry | Creating AI agents with MCP tool access |
| Cleanup | Deleting resources to avoid charges |

## What's Next?

You now have a complete understanding of MCP from local development to cloud deployment. Ideas for continuing:

- Add authentication to your MCP server (API keys, Entra ID)
- Create a CI/CD pipeline for automatic deployment
- Build more tools tailored to your team's workflow
- Connect your MCP server to multiple AI clients simultaneously